# 02 — Models and deployment benchmark

Trains every model on every prepared dataset, exports the transformers to ONNX (fp32 and int8), then measures **quality and deployment cost of the same artefact** on a single CPU thread: size on disk, peak memory, cold-start time, latency p50/p95, throughput, CPU time per text. Produces the paper's tables, the Pareto figure and the deployment-scenario table.

| Stage | What | Runtime | Resumable |
|---|---|---|---|
| A | Classical models (NB, SVM, LR on TF-IDF) | **CPU** runtime, ~10 min | yes |
| B | Transformer fine-tuning, seeds × datasets × models | **GPU (T4)**, ~4–5 h total over 2 days | yes, per run |
| C | ONNX export + int8 quantisation + equivalence checks (`BENCH_DATASETS` only) | CPU | yes |
| D | CPU benchmark, one fresh process per artefact (`BENCH_DATASETS` only) | **CPU** runtime, ~1 h | yes, per artefact |
| E | Tables, Pareto figure, scenarios, TWIFIL shortcut analysis, bootstrap CIs | CPU, 1 min | — |

**How to run it (nothing to edit):** with `RUN_STAGES = "auto"`, a **T4** runtime runs stage B only; a **CPU** runtime runs A, C and D (D waits until every model is trained, so all models are benchmarked in one session on the same CPU). So: *Run all* on T4 each day until it prints `0 runs to do`, then *Run all* once on CPU. Every finished run is saved to Drive immediately; B stops launching new runs after `GPU_TIME_BUDGET_H`, so re-run the notebook the next day and it continues where it stopped. Training is seed-major: after the first pass you already have one complete seed for every model and dataset.

**Optional smoke test (≈20 min on T4):** `SMOKE_TEST = True` and `RUN_STAGES = "ABCDE"`, *Run all*, check that every stage passes, then restore `False` / `"auto"`. Smoke-test outputs go to separate `*_smoke` folders.

Requires the outputs of `01_data_preparation.ipynb` in `MyDrive/alg_sentiment_benchmark/processed/`.

## 1. Configuration

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/alg_sentiment_benchmark"
SMOKE_TEST = False            # True: tiny subsets, 1 seed, 1 epoch, separate *_smoke folders
RUN_STAGES = "auto"           # "auto": T4 runtime -> stage B only; CPU runtime -> A, C, D (D waits until all models are trained).
                              # Or set explicitly, e.g. "ABCDE" for the smoke test. Stage E (tables) always runs.

# datasets from notebook 01 (missing ones are skipped); order = training priority
DATASETS = ["twifil_clean", "youtube", "twifil_official", "narabizi"]

# ---- transformers (Hugging Face ids) ----
TRANSFORMERS = {
    "dziribert":    {"hf": "alger-ia/dziribert",                      "lr": 3e-5},
    "marbertv2":    {"hf": "UBC-NLP/MARBERTv2",                       "lr": 3e-5},
    "camelbert_da": {"hf": "CAMeL-Lab/bert-base-arabic-camelbert-da", "lr": 3e-5},
    "me5_small":    {"hf": "intfloat/multilingual-e5-small",          "lr": 5e-5},
}
SEEDS = [13, 42, 87]          # 3 seeds; add more if time allows
EPOCHS = 4                    # max epochs; early stopping on dev macro-F1
PATIENCE = 1
BATCH_SIZE = 32
MAX_LEN_CAP = 128             # max_len = 99th percentile of train token lengths, capped here
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01
GPU_TIME_BUDGET_H = 3.4       # stage B stops starting new runs after this many hours in the session

# ---- storage: models are saved, exported and benchmarked only for these datasets ----
# (deployment cost depends on the architecture, not the dataset; quality runs happen for every dataset)
BENCH_DATASETS = ["twifil_clean", "youtube"]
KEEP_ONNX_FP32 = False        # delete the fp32 .onnx once benchmarked (saves ≈2 GB per dataset; int8 is kept)
ALLOW_CPU_TRAINING = False    # stage B refuses to train on CPU unless True

# ---- classical models: hyper-parameter grids (selected on dev macro-F1) ----
CLASSICAL_GRIDS = {
    "nb_char":      {"alpha": [0.01, 0.03, 0.1, 0.3, 1.0]},
    "svm_char":     {"C": [0.03, 0.1, 0.3, 1.0]},
    "lr_word_char": {"C": [0.25, 1.0, 4.0, 16.0, 64.0]},
}

# ---- CPU benchmark ----
BENCH_THREADS = 1             # 1 thread = edge-like; the same for every model
N_SINGLE = 300                # texts timed one by one (batch = 1) for p50/p95
BENCH_ROUNDS = 3              # the single-text timing is repeated; tables report the median round and the spread
TIMING_REPEATS = 3            # extra timing-only passes over ALL artefacts, interleaved in time (robust to VM contention)
N_TIMING = 200                # texts per timing-only pass
WARMUP_CALLS = 20
BENCH_BATCH = 32              # batch size for the throughput pass over the full test set
BENCH_TIMEOUT_S = 3600

# ---- deployment scenarios: (max size MB, max p95 latency ms on the benchmark CPU) ----
# No physical device is used: budgets are expressed on the benchmark CPU and named neutrally.
SCENARIOS = {
    "Tight budget (≤ 25 MB, p95 ≤ 20 ms)":      (25, 20),
    "Moderate budget (≤ 150 MB, p95 ≤ 100 ms)": (150, 100),
    "No constraint":                            (float("inf"), float("inf")),
}
N_BOOTSTRAP = 1000

# ---- names and families used in tables and figures ----
DISPLAY = {"nb_char": "NB-char", "svm_char": "SVM-char", "lr_word_char": "LR-word+char",
           "dziribert": "DziriBERT", "marbertv2": "MARBERTv2", "camelbert_da": "CAMeLBERT-DA", "me5_small": "mE5-small"}
FAMILY = {"nb_char": "Classical (TF-IDF)", "svm_char": "Classical (TF-IDF)", "lr_word_char": "Classical (TF-IDF)",
          "me5_small": "Multilingual small transformer",
          "dziribert": "Arabic/dialect BERT", "marbertv2": "Arabic/dialect BERT", "camelbert_da": "Arabic/dialect BERT"}

## 2. Environment

In [ ]:
import os, sys, json, time, math, shutil, subprocess, platform, random, copy, datetime
SESSION_START = time.time()

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount("/content/drive")
else:
    PROJECT_DIR = os.environ.get("ALG_PROJECT_DIR", os.path.abspath("alg_sentiment_benchmark"))

# prepared data shipped as a zip: drop alg_sentiment_processed.zip in MyDrive; any dataset missing from
# processed/ is unpacked from it (datasets already present are left untouched)
import zipfile
_parent = os.path.dirname(PROJECT_DIR.rstrip("/"))
for _cand in (os.path.join(_parent, "alg_sentiment_processed.zip"), os.path.join(PROJECT_DIR, "alg_sentiment_processed.zip")):
    if os.path.exists(_cand):
        with zipfile.ZipFile(_cand) as zf:
            _members = [m for m in zf.namelist() if m.startswith("alg_sentiment_benchmark/processed/") and m.count("/") >= 3]
            _in_zip = sorted({m.split("/")[2] for m in _members if m.split("/")[2]})
            _missing = [d for d in _in_zip if not os.path.exists(os.path.join(PROJECT_DIR, "processed", d, "test.csv"))]
            if _missing:
                zf.extractall(_parent, members=[m for m in _members if m.split("/")[2] in _missing])
                print("unpacked from", os.path.basename(_cand), "->", _missing)
        break


def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


for mod, pkg in (("onnx", "onnx"), ("onnxruntime", "onnxruntime"), ("psutil", "psutil")):
    try:
        __import__(mod)
    except ImportError:
        _pip(pkg)

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.metrics import f1_score, accuracy_score

suffix = "_smoke" if SMOKE_TEST else ""
PROC_DIR = os.path.join(PROJECT_DIR, "processed")
CODE_DIR = os.path.join(PROJECT_DIR, "code")
RUNS_DIR = os.path.join(PROJECT_DIR, "runs" + suffix)
MODELS_DIR = os.path.join(PROJECT_DIR, "models" + suffix)
BENCH_DIR = os.path.join(PROJECT_DIR, "bench" + suffix)
REPORT_DIR = os.path.join(PROJECT_DIR, "reports" + suffix)
for d in (CODE_DIR, RUNS_DIR, MODELS_DIR, BENCH_DIR, REPORT_DIR):
    os.makedirs(d, exist_ok=True)

if SMOKE_TEST:
    SEEDS, EPOCHS = SEEDS[:1], 1
    CLASSICAL_GRIDS = {k: {p: v[:1] for p, v in g.items()} for k, g in CLASSICAL_GRIDS.items()}
    N_SINGLE, WARMUP_CALLS = 30, 3

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else None
except ImportError:
    torch, DEVICE, GPU_NAME = None, "cpu", None


def cpu_name():
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "unknown"


def cpu_flags():
    """ISA extensions that decide int8 kernel behaviour (VNNI avoids the U8S8 saturation issue)."""
    try:
        with open("/proc/cpuinfo") as f:
            flags = next((l.split(":", 1)[1].split() for l in f if l.startswith("flags")), [])
    except OSError:
        flags = []
    return [x for x in ("avx2", "avx512f", "avx512_vnni", "avx_vnni", "amx_int8") if x in flags]


print(f"Colab: {IN_COLAB} | device: {DEVICE} {GPU_NAME or ''} | CPU: {cpu_name()} ({os.cpu_count()} vCPU) | flags: {cpu_flags()}")
print(f"Python {platform.python_version()} | torch {getattr(torch, '__version__', '-')} | scikit-learn {sklearn.__version__}")
print("Project dir:", PROJECT_DIR, "| smoke test:", SMOKE_TEST)
AUTO_STAGES = RUN_STAGES == "auto"
if AUTO_STAGES:
    RUN_STAGES = "B" if DEVICE == "cuda" else "ACD"
print(f"Stages this session: {RUN_STAGES} + E" + (" (auto)" if AUTO_STAGES else ""))
try:
    _du = shutil.disk_usage(PROJECT_DIR)
    print(f"Free space at the project folder: {_du.free / 2**30:.1f} GB (budget ≈3 GB per benchmarked dataset)")
except OSError:
    pass

### Helper modules
Written to `code/` in the project folder: `algsent_utils.py` (text normalisation for the classical models; must be importable wherever a saved classical model is loaded) and `bench_worker.py` (the CPU benchmark, run in a fresh process per artefact so that memory and cold-start figures belong to that model alone).

In [ ]:
UTILS_SRC = r"""
# Shared text normalisation for the classical (TF-IDF) models.
import re

_URL = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
_MENTION = re.compile(r"@\w+")
_DIAC = re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۭ]")
_ELONG = re.compile(r"(.)\1{2,}")
_AR = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ى": "ي", "ة": "ه", "ؤ": "و", "ئ": "ي", "ـ": ""})


def normalise(text):
    s = str(text).lower()
    s = _URL.sub(" httpurl ", s)       # kept as a token: official TWIFIL still contains URLs
    s = _MENTION.sub(" @user ", s)
    s = _DIAC.sub("", s).translate(_AR)
    s = _ELONG.sub(r"\1\1", s)
    return " ".join(s.split())


def normalise_batch(texts):
    return [normalise(t) for t in texts]
"""

WORKER_SRC = r"""
# CPU benchmark worker. One fresh process per artefact:
#   python bench_worker.py --spec spec.json --texts texts.json --out result.json --threads 1
import argparse, json, os, sys, time

ap = argparse.ArgumentParser()
ap.add_argument("--spec", required=True)
ap.add_argument("--texts", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--threads", type=int, default=1)
ap.add_argument("--n_single", type=int, default=300)
ap.add_argument("--warmup", type=int, default=20)
ap.add_argument("--batch", type=int, default=32)
ap.add_argument("--rounds", type=int, default=3)
ap.add_argument("--timing_only", action="store_true")   # skip the full-test pass (used for the repeated timing passes)
a = ap.parse_args()
for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = str(a.threads)
os.environ["CUDA_VISIBLE_DEVICES"] = ""          # CPU only, even on a GPU runtime
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))


def rss_mb():
    with open("/proc/self/statm") as f:
        return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2 ** 20


def peak_rss_mb():
    # VmHWM resets at exec; ru_maxrss would inherit the notebook's own peak through fork + exec
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmHWM:"):
                return int(line.split()[1]) / 1024
    return float("nan")


spec = json.load(open(a.spec, encoding="utf-8"))
texts = json.load(open(a.texts, encoding="utf-8"))
rss_start = rss_mb()
t0 = time.perf_counter()                          # cold start = library import + model load

if spec["kind"] == "sklearn":
    import joblib
    model = joblib.load(spec["path"])

    def predict(batch):
        return [int(x) for x in model.predict(batch)]

elif spec["kind"] == "torch":
    import torch
    torch.set_num_threads(a.threads)
    torch.set_num_interop_threads(1)
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(spec["path"])
    model = AutoModelForSequenceClassification.from_pretrained(spec["path"]).eval()

    def predict(batch):
        enc = tok(batch, truncation=True, max_length=spec["max_len"], padding=True, return_tensors="pt")
        with torch.inference_mode():
            return model(**enc).logits.argmax(-1).tolist()

elif spec["kind"] == "onnx":
    import numpy as np
    import onnxruntime as ort
    from tokenizers import Tokenizer
    so = ort.SessionOptions()
    so.intra_op_num_threads, so.inter_op_num_threads = a.threads, 1
    sess = ort.InferenceSession(spec["path"], so, providers=["CPUExecutionProvider"])
    tk = Tokenizer.from_file(spec["tokenizer"])
    tk.enable_truncation(max_length=spec["max_len"])
    tk.enable_padding(pad_id=spec["pad_id"], pad_token=spec["pad_token"])
    names = [i.name for i in sess.get_inputs()]

    def predict(batch):
        encs = tk.encode_batch(batch)
        feed = {"input_ids": np.array([e.ids for e in encs], dtype=np.int64),
                "attention_mask": np.array([e.attention_mask for e in encs], dtype=np.int64),
                "token_type_ids": np.array([e.type_ids for e in encs], dtype=np.int64)}
        return sess.run(None, {k: feed[k] for k in names})[0].argmax(-1).tolist()
else:
    raise ValueError(spec["kind"])

load_s = time.perf_counter() - t0
rss_loaded = rss_mb()

step = max(1, len(texts) // a.n_single)
sample = texts[::step][: a.n_single]
for t in sample[: a.warmup]:
    predict([t])
def pct(values, p):
    v = sorted(values)
    return v[min(len(v) - 1, int(round(p * (len(v) - 1))))]


rounds = []                                       # the same sample timed in several rounds
for _ in range(a.rounds):
    lat = []
    for t in sample:
        t1 = time.perf_counter()
        predict([t])
        lat.append((time.perf_counter() - t1) * 1000)
    rounds.append({"p50": pct(lat, 0.50), "p95": pct(lat, 0.95), "mean": sum(lat) / len(lat)})

preds, c0, w0 = [], time.process_time(), time.perf_counter()
if not a.timing_only:
    for i in range(0, len(texts), a.batch):
        preds += predict(texts[i:i + a.batch])
wall, cpu = time.perf_counter() - w0, time.process_time() - c0

med = lambda k: pct([r[k] for r in rounds], 0.5)
p50s = [r["p50"] for r in rounds]
out = {"n_texts": len(texts), "n_single": len(sample), "rounds": rounds, "threads": a.threads,
       "load_s": load_s, "rss_start_mb": rss_start, "rss_loaded_mb": rss_loaded,
       "peak_rss_mb": peak_rss_mb(),
       "p50_ms": med("p50"), "p95_ms": med("p95"), "mean_ms": med("mean"),
       "p50_spread_pct": 100 * (max(p50s) - min(p50s)) / med("p50"),
       "throughput_per_s": None if a.timing_only else len(texts) / wall,
       "cpu_ms_per_text": None if a.timing_only else 1000 * cpu / len(texts), "preds": preds}
json.dump(out, open(a.out, "w", encoding="utf-8"))
"""

for name, src in (("algsent_utils.py", UTILS_SRC), ("bench_worker.py", WORKER_SRC)):
    with open(os.path.join(CODE_DIR, name), "w", encoding="utf-8") as f:
        f.write(src.lstrip())
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import importlib, algsent_utils
importlib.reload(algsent_utils)
from algsent_utils import normalise, normalise_batch
assert normalise("ههههه رائعةةة @x https://t.co/a") == "هه رايعهه @user httpurl"
print("helper modules written to", CODE_DIR)

## 3. Data

In [ ]:
LABEL_NAMES = {0: "negative", 1: "neutral", 2: "positive", 3: "mixed"}


def load_dataset(ds):
    parts = {}
    for sp in ("train", "dev", "test"):
        df = pd.read_csv(os.path.join(PROC_DIR, ds, f"{sp}.csv"), keep_default_na=False, dtype={"text": str})
        df["label"] = df["label"].astype(int)
        if SMOKE_TEST:                                     # small stratified subset
            n = {"train": 240, "dev": 80, "test": 80}[sp]
            df = df.sample(frac=1, random_state=0).groupby("label").head(n // df["label"].nunique() + 1)
            df = df.head(n).reset_index(drop=True)
        parts[sp] = df
    return parts


DATA = {}
for ds in DATASETS:
    if all(os.path.exists(os.path.join(PROC_DIR, ds, f"{sp}.csv")) for sp in ("train", "dev", "test")):
        DATA[ds] = load_dataset(ds)
    else:
        print(f"skip {ds}: not prepared (run notebook 01)")
N_LABELS = {ds: int(max(p["train"]["label"].max(), p["test"]["label"].max()) + 1) for ds, p in DATA.items()}
for ds, p in DATA.items():
    print(f"{ds:16s} train {len(p['train']):6d} | dev {len(p['dev']):5d} | test {len(p['test']):5d} | labels {N_LABELS[ds]}")


def scores(y, p):
    y, p = np.asarray(y), np.asarray(p)
    labs = sorted(set(y))
    per = f1_score(y, p, labels=labs, average=None, zero_division=0)
    return {"macro_f1": float(f1_score(y, p, labels=labs, average="macro", zero_division=0)),
            "accuracy": float(accuracy_score(y, p)),
            "per_class_f1": {LABEL_NAMES[l]: float(v) for l, v in zip(labs, per)}}


def run_dir(ds, model, seed):
    return os.path.join(RUNS_DIR, ds, model, f"seed{seed}")


def save_run(ds, model, seed, metrics, test_preds):
    d = run_dir(ds, model, seed)
    os.makedirs(d, exist_ok=True)
    pd.DataFrame({"id": DATA[ds]["test"]["id"], "label": DATA[ds]["test"]["label"], "pred": test_preds}) \
        .to_csv(os.path.join(d, "test_preds.csv"), index=False)
    with open(os.path.join(d, "metrics.json"), "w") as f:     # written last = marks the run as complete
        json.dump(metrics, f, indent=2)

## 4. Stage A — classical models (CPU)

Char n-grams (`char_wb`, 2–5) are robust to Arabizi, elongation and code-switching; the LR model adds word uni/bi-grams. Hyper-parameters are chosen on dev macro-F1; the selected model is refit on train only (the same data budget as the transformers, which use dev for early stopping). These models are deterministic, so one run each.

In [ ]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression


def char_tfidf():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=300_000)


def make_classical(name, **hp):
    norm = ("norm", FunctionTransformer(normalise_batch))
    if name == "nb_char":
        return Pipeline([norm, ("tfidf", char_tfidf()), ("clf", ComplementNB(alpha=hp["alpha"]))])
    if name == "svm_char":
        return Pipeline([norm, ("tfidf", char_tfidf()), ("clf", LinearSVC(C=hp["C"], class_weight="balanced"))])
    if name == "lr_word_char":
        feats = FeatureUnion([
            ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200_000,
                                     token_pattern=r"(?u)\b\w+\b")),
            ("char", char_tfidf())])
        return Pipeline([norm, ("feats", feats),
                         ("clf", LogisticRegression(C=hp["C"], max_iter=3000, class_weight="balanced"))])
    raise ValueError(name)


def grid(g):
    keys = list(g)
    for vals in __import__("itertools").product(*(g[k] for k in keys)):
        yield dict(zip(keys, vals))


for ds, parts in (DATA.items() if "A" in RUN_STAGES else []):
    tr, dv, te = parts["train"], parts["dev"], parts["test"]
    for name, g in CLASSICAL_GRIDS.items():
        if os.path.exists(os.path.join(run_dir(ds, name, 0), "metrics.json")):
            print(f"[done] {ds} / {name}")
            continue
        best = None
        for hp in grid(g):
            m = make_classical(name, **hp).fit(tr["text"].tolist(), tr["label"])
            f = scores(dv["label"], m.predict(dv["text"].tolist()))["macro_f1"]
            if best is None or f > best[0]:
                best = (f, hp, m)
        f_dev, hp, m = best
        edge = [k for k, v in hp.items() if len(g[k]) > 1 and v in (min(g[k]), max(g[k]))]
        if edge:
            print(f"  note: {ds}/{name} picked a grid edge for {edge}; consider widening CLASSICAL_GRIDS")
        t0 = time.perf_counter()
        pred = m.predict(te["text"].tolist())
        mdir = os.path.join(MODELS_DIR, ds, name)
        os.makedirs(mdir, exist_ok=True)
        joblib.dump(m, os.path.join(mdir, "model.joblib"))
        metrics = {"dataset": ds, "model": name, "seed": 0, "hyperparameters": hp, "dev_macro_f1": f_dev,
                   **scores(te["label"], pred)}
        save_run(ds, name, 0, metrics, pred)
        print(f"{ds:16s} {name:13s} {hp} dev F1 {f_dev:.3f} | test F1 {metrics['macro_f1']:.3f} acc {metrics['accuracy']:.3f}")

## 5. Stage B — transformer fine-tuning (GPU)

Plain PyTorch loop (no `Trainer`, so it does not break across `transformers` versions): AdamW, linear warm-up/decay, fp16 autocast, early stopping on dev macro-F1, best epoch restored. Only the **first seed's** model is saved, and only for `BENCH_DATASETS` (it is the one exported and benchmarked; latency does not depend on the seed or the dataset); every other run saves metrics and predictions only. Drive usage: ≈2 GB of PyTorch weights plus ≈1 GB of int8 ONNX per benchmarked dataset.

In [ ]:
import warnings
if torch is not None:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from transformers.utils import logging as hf_logging
    from torch.utils.data import DataLoader
    hf_logging.set_verbosity_error()          # hide load reports ("classifier newly initialised" is expected)
    hf_logging.disable_progress_bar()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_max_len(tok, texts):
    lens = [len(x) for x in tok(list(texts), truncation=False)["input_ids"]]
    return int(min(MAX_LEN_CAP, max(16, np.percentile(lens, 99))))


def encode(tok, df, max_len):
    enc = tok(df["text"].tolist(), truncation=True, max_length=max_len)
    keys = [k for k in ("input_ids", "attention_mask", "token_type_ids") if k in enc]
    return [{**{k: enc[k][i] for k in keys}, "labels": int(y)} for i, y in enumerate(df["label"])]


def loader(tok, feats, shuffle, seed=0):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(feats, batch_size=BATCH_SIZE if shuffle else 128, shuffle=shuffle, generator=g,
                      collate_fn=lambda b: tok.pad(b, return_tensors="pt"))


@torch.no_grad() if torch is not None else (lambda f: f)
def predict_loader(model, dl):
    model.eval()
    out = []
    for b in dl:
        b = {k: v.to(DEVICE) for k, v in b.items() if k != "labels"}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            out += model(**b).logits.argmax(-1).tolist()
    return out


def train_one(ds, name, seed, save_model):
    cfg, parts = TRANSFORMERS[name], DATA[ds]
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(cfg["hf"])
    model = AutoModelForSequenceClassification.from_pretrained(cfg["hf"], num_labels=N_LABELS[ds]).to(DEVICE)
    max_len = pick_max_len(tok, parts["train"]["text"])
    tr = loader(tok, encode(tok, parts["train"], max_len), True, seed)
    dv = loader(tok, encode(tok, parts["dev"], max_len), False)
    te = loader(tok, encode(tok, parts["test"], max_len), False)

    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    groups = [{"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=cfg["lr"])
    total = EPOCHS * len(tr)
    warm = int(WARMUP_RATIO * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / max(1, warm) if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")

    best_f1, best_state, best_epoch, bad, history = -1.0, None, 0, 0, []
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for b in tr:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
                loss = model(**b).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
        f_dev = scores(parts["dev"]["label"], predict_loader(model, dv))["macro_f1"]
        history.append({"epoch": epoch, "dev_macro_f1": f_dev})
        if f_dev > best_f1:
            best_f1, best_epoch, bad = f_dev, epoch, 0
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad > PATIENCE:
                break
    model.load_state_dict(best_state)
    pred = predict_loader(model, te)
    metrics = {"dataset": ds, "model": name, "seed": seed, "hf_id": cfg["hf"], "lr": cfg["lr"], "max_len": max_len,
               "best_epoch": best_epoch, "dev_macro_f1": best_f1, "history": history,
               "train_minutes": (time.time() - t0) / 60, "device": GPU_NAME or DEVICE, **scores(parts["test"]["label"], pred)}
    if save_model:
        mdir = os.path.join(MODELS_DIR, ds, name, "hf")
        model.save_pretrained(mdir, safe_serialization=True)
        tok.save_pretrained(mdir)
        with open(os.path.join(mdir, "bench_meta.json"), "w") as f:
            json.dump({"max_len": max_len, "pad_id": tok.pad_token_id, "pad_token": tok.pad_token}, f)
    save_run(ds, name, seed, metrics, pred)
    del model, opt, best_state
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return metrics


def stage_b():
    if torch is None:
        print("PyTorch not installed; skipping stage B")
        return
    if DEVICE != "cuda" and not ALLOW_CPU_TRAINING:
        print("Stage B needs a GPU runtime (Runtime > Change runtime type > T4). Skipping.")
        return
    todo = [(s, ds, m) for s in SEEDS for ds in DATA for m in TRANSFORMERS
            if not os.path.exists(os.path.join(run_dir(ds, m, s), "metrics.json"))]
    print(f"{len(todo)} runs to do")
    for i, (seed, ds, name) in enumerate(todo, 1):
        hours = (time.time() - SESSION_START) / 3600
        if hours > GPU_TIME_BUDGET_H:
            print(f"Time budget reached ({hours:.2f} h). {len(todo) - i + 1} runs left: re-run tomorrow.")
            break
        m = train_one(ds, name, seed, save_model=(seed == SEEDS[0] and ds in BENCH_DATASETS))
        print(f"[{i}/{len(todo)}] {ds:16s} {name:13s} seed {seed:3d} | test F1 {m['macro_f1']:.3f} acc {m['accuracy']:.3f} "
              f"| best epoch {m['best_epoch']} | {m['train_minutes']:.1f} min")


if "B" in RUN_STAGES:
    stage_b()
else:
    print("stage B skipped (RUN_STAGES)")

## 6. Stage C — ONNX export and int8 quantisation (CPU)

For each saved transformer: export to ONNX (fp32, opset 17, dynamic batch and sequence length), check that the ONNX logits match PyTorch (max |Δ| < 1e-3) and that the standalone `tokenizers` pipeline used in deployment produces the same ids as `AutoTokenizer`, then apply dynamic int8 quantisation to the weights.

In [ ]:
def dir_size_mb(path, files=None):
    if os.path.isfile(path):
        return os.path.getsize(path) / 2 ** 20
    names = files if files is not None else os.listdir(path)
    return sum(os.path.getsize(os.path.join(path, n)) for n in names if os.path.isfile(os.path.join(path, n))) / 2 ** 20


class _Logits(torch.nn.Module if torch is not None else object):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        return self.m(**kw).logits


def export_onnx(hf_dir, out_path):
    tok = AutoTokenizer.from_pretrained(hf_dir)
    model = AutoModelForSequenceClassification.from_pretrained(hf_dir).eval()
    enc = tok(["مثال قصير", "exemple un peu plus long pour la forme"], padding=True, return_tensors="pt")
    names = [k for k in ("input_ids", "attention_mask", "token_type_ids") if k in enc]
    args = tuple(enc[k] for k in names)
    dyn = {k: {0: "batch", 1: "seq"} for k in names}
    dyn["logits"] = {0: "batch"}
    kw = dict(input_names=names, output_names=["logits"], dynamic_axes=dyn, opset_version=17)
    wrapper = _Logits(model).eval()      # a new Module starts in train mode; the exporter would restore that mode
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            torch.onnx.export(wrapper, args, out_path, dynamo=False, **kw)
    except TypeError:                                   # older torch without the dynamo argument
        torch.onnx.export(wrapper, args, out_path, **kw)
    except Exception as e:                              # future torch without the legacy exporter
        print("legacy ONNX exporter failed, using the dynamo exporter:", str(e)[:150])
        try:
            import onnxscript  # noqa: F401
        except ImportError:
            _pip("onnxscript")
        b, q = torch.export.Dim("batch"), torch.export.Dim("seq", max=512)
        shapes = tuple({0: b, 1: q} for _ in names)
        torch.onnx.export(wrapper, args, out_path, dynamo=True, input_names=names, output_names=["logits"],
                          dynamic_shapes=shapes, opset_version=18)
    model.eval()
    return tok, model, names


def check_equivalence(hf_dir, onnx_path, tok, model, texts, meta):
    import onnxruntime as ort
    from tokenizers import Tokenizer
    tk = Tokenizer.from_file(os.path.join(hf_dir, "tokenizer.json"))
    tk.enable_truncation(max_length=meta["max_len"])
    tk.enable_padding(pad_id=meta["pad_id"], pad_token=meta["pad_token"])
    encs = tk.encode_batch(texts)
    ref = tok(texts, truncation=True, max_length=meta["max_len"], padding=True, return_tensors="pt")
    assert [e.ids for e in encs] == ref["input_ids"].tolist(), "tokenizers vs AutoTokenizer ids differ"
    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    feed = {"input_ids": np.array([e.ids for e in encs], dtype=np.int64),
            "attention_mask": np.array([e.attention_mask for e in encs], dtype=np.int64),
            "token_type_ids": np.array([e.type_ids for e in encs], dtype=np.int64)}
    got = sess.run(None, {i.name: feed[i.name] for i in sess.get_inputs()})[0]
    with torch.inference_mode():
        want = model(**ref).logits.numpy()
    return float(np.abs(got - want).max())


if torch is not None and "C" in RUN_STAGES:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    for ds in [d for d in DATA if d in BENCH_DATASETS]:
        for name in TRANSFORMERS:
            hf_dir = os.path.join(MODELS_DIR, ds, name, "hf")
            if not os.path.exists(os.path.join(hf_dir, "bench_meta.json")):
                continue
            base = os.path.join(MODELS_DIR, ds, name)
            fp32, int8, int8rr = (os.path.join(base, f) for f in ("model.onnx", "model.int8.onnx", "model.int8pcrr.onnx"))
            fp32_benched = os.path.exists(os.path.join(BENCH_DIR, ds, f"{name}__ONNX_fp32.json"))
            todo_q = [p for p in (int8, int8rr) if not os.path.exists(p)]
            if not todo_q and (os.path.exists(fp32) or fp32_benched):
                print(f"[done] {ds} / {name}")
                continue
            msg = ""
            if not os.path.exists(fp32):
                meta = json.load(open(os.path.join(hf_dir, "bench_meta.json")))
                tok, model, names = export_onnx(hf_dir, fp32)
                diff = check_equivalence(hf_dir, fp32, tok, model, DATA[ds]["dev"]["text"].head(16).tolist(), meta)
                assert diff < 1e-3, f"{ds}/{name}: ONNX fp32 differs from PyTorch (max |Δ| = {diff:.2e})"
                msg = f"inputs {names} | max |Δ logits| {diff:.1e} | "
                del model
            for out in todo_q:
                # default call (per-tensor, full range) vs the ONNX Runtime recommendation for AVX2/AVX512 CPUs
                # without VNNI: per-channel scales + reduce_range (7-bit weights) to avoid U8S8 saturation
                rr = out == int8rr
                quantize_dynamic(fp32, out, weight_type=QuantType.QInt8, per_channel=rr, reduce_range=rr)
            print(f"{ds:16s} {name:13s} {msg}fp32 {dir_size_mb(fp32):.0f} MB -> int8 {dir_size_mb(int8):.0f} MB, "
                  f"int8 per-channel+RR {dir_size_mb(int8rr):.0f} MB")
            if fp32_benched and not KEEP_ONNX_FP32:
                os.remove(fp32)       # already benchmarked earlier; only needed here to quantise

## 7. Stage D — CPU benchmark (one fresh process per artefact)

Artefacts: every classical model (scikit-learn), and every saved transformer in four formats (PyTorch fp32, ONNX fp32, ONNX int8 with the default call, ONNX int8 with per-channel scales + reduce-range). Each worker loads the artefact, times `N_SINGLE` single-text predictions (after warm-up), then predicts the **whole test set** in batches; quality in the deployment table comes from these predictions, so quality and cost are measured on the same artefact. All numbers use `BENCH_THREADS` threads. After the main pass, `TIMING_REPEATS` timing-only passes visit every artefact again in turn, so each latency is the median of several measurements taken at different times (Colab VMs share physical CPUs, so contention drifts during a long session). Absolute milliseconds depend on the Colab CPU; the comparison between models is what the paper should rely on. Run the whole stage in one runtime type so that every model sees the same CPU.

In [ ]:
def artefacts(ds):
    out = []
    for name in CLASSICAL_GRIDS:
        p = os.path.join(MODELS_DIR, ds, name, "model.joblib")
        if os.path.exists(p):
            out.append({"model": name, "format": "scikit-learn", "kind": "sklearn", "path": p, "size_mb": dir_size_mb(p)})
    for name in TRANSFORMERS:
        base = os.path.join(MODELS_DIR, ds, name)
        hf = os.path.join(base, "hf")
        if not os.path.exists(os.path.join(hf, "bench_meta.json")):
            continue
        meta = json.load(open(os.path.join(hf, "bench_meta.json")))
        tok_json = os.path.join(hf, "tokenizer.json")
        out.append({"model": name, "format": "PyTorch fp32", "kind": "torch", "path": hf, "size_mb": dir_size_mb(hf), **meta})
        for fmt, fn in (("ONNX fp32", "model.onnx"), ("ONNX int8", "model.int8.onnx"),
                        ("ONNX int8 per-channel+RR", "model.int8pcrr.onnx")):
            p = os.path.join(base, fn)
            if os.path.exists(p):
                out.append({"model": name, "format": fmt, "kind": "onnx", "path": p, "tokenizer": tok_json,
                            "size_mb": dir_size_mb(p) + dir_size_mb(tok_json), **meta})
    return out


def bench_path(ds, a):
    return os.path.join(BENCH_DIR, ds, f"{a['model']}__{a['format'].replace(' ', '_')}.json")


BENCH_CPU = cpu_name()
_missing = [f"{ds}/{m}" for ds in BENCH_DATASETS if ds in DATA for m in TRANSFORMERS
            if not os.path.exists(os.path.join(MODELS_DIR, ds, m, "hf", "bench_meta.json"))]
if "D" in RUN_STAGES and AUTO_STAGES and _missing:
    RUN_STAGES = RUN_STAGES.replace("D", "")
    print(f"Stage D postponed: {len(_missing)} model(s) not trained yet ({', '.join(_missing[:4])}...). "
          "Finish stage B on a T4 runtime, then run the notebook once more on CPU, so that every model is "
          "benchmarked in the same session on the same CPU.")
for ds, parts in [(d, p) for d, p in DATA.items() if d in BENCH_DATASETS and "D" in RUN_STAGES]:
    os.makedirs(os.path.join(BENCH_DIR, ds), exist_ok=True)
    texts_path = os.path.join(BENCH_DIR, ds, "_test_texts.json")
    with open(texts_path, "w", encoding="utf-8") as f:
        json.dump(parts["test"]["text"].tolist(), f, ensure_ascii=False)
    for a in artefacts(ds):
        out = bench_path(ds, a)
        if os.path.exists(out):
            print(f"[done] {ds} / {a['model']} / {a['format']}")
            continue
        spec = out.replace(".json", ".spec.json")
        with open(spec, "w") as f:
            json.dump(a, f)
        tmp = out + ".tmp"
        res = subprocess.run([sys.executable, os.path.join(CODE_DIR, "bench_worker.py"), "--spec", spec, "--texts", texts_path,
                              "--out", tmp, "--threads", str(BENCH_THREADS), "--n_single", str(N_SINGLE),
                              "--warmup", str(WARMUP_CALLS), "--batch", str(BENCH_BATCH), "--rounds", str(BENCH_ROUNDS)],
                             capture_output=True, text=True, timeout=BENCH_TIMEOUT_S)
        if res.returncode != 0:
            print(f"FAILED {ds} / {a['model']} / {a['format']}\n{res.stderr[-2000:]}")
            continue
        r = json.load(open(tmp, encoding="utf-8"))
        r.update({"dataset": ds, "model": a["model"], "format": a["format"], "size_mb": a["size_mb"], "cpu": BENCH_CPU,
                  "cpu_flags": cpu_flags(), "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
                  **scores(parts["test"]["label"], r["preds"])})
        with open(out, "w", encoding="utf-8") as f:
            json.dump(r, f)
        os.remove(tmp)
        if a["format"] == "ONNX fp32" and not KEEP_ONNX_FP32:
            os.remove(a["path"])                              # benchmarked; the int8 model is the one kept
        print(f"{ds:16s} {DISPLAY[a['model']]:13s} {a['format']:13s} F1 {r['macro_f1']:.3f} | {r['size_mb']:7.1f} MB | "
              f"RSS {r['peak_rss_mb']:6.0f} MB | load {r['load_s']:5.1f} s | p50 {r['p50_ms']:7.2f} ms | "
              f"p95 {r['p95_ms']:7.2f} ms | {r['throughput_per_s']:7.1f} texts/s")

# ---- repeated timing passes, interleaved over all artefacts (each pass visits every model once) ----
if "D" in RUN_STAGES:
    for rep in range(1, TIMING_REPEATS + 1):
        n_new = 0
        for ds in [d for d in DATA if d in BENCH_DATASETS]:
            texts_path = os.path.join(BENCH_DIR, ds, "_test_texts.json")
            for a in artefacts(ds):
                main = bench_path(ds, a)
                out = main.replace(".json", f".t{rep}.json")
                if not os.path.exists(main) or os.path.exists(out):
                    continue
                spec = main.replace(".json", ".spec.json")
                res = subprocess.run([sys.executable, os.path.join(CODE_DIR, "bench_worker.py"), "--spec", spec, "--texts", texts_path,
                                      "--out", out + ".tmp", "--threads", str(BENCH_THREADS), "--n_single", str(N_TIMING),
                                      "--warmup", "10", "--rounds", "1", "--timing_only"],
                                     capture_output=True, text=True, timeout=BENCH_TIMEOUT_S)
                if res.returncode != 0:
                    print(f"FAILED timing pass {rep}: {ds} / {a['model']} / {a['format']}\n{res.stderr[-1000:]}")
                    continue
                r = json.load(open(out + ".tmp", encoding="utf-8"))
                r.update({"cpu": BENCH_CPU, "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")})
                with open(out, "w", encoding="utf-8") as f:
                    json.dump(r, f)
                os.remove(out + ".tmp")
                n_new += 1
        print(f"timing pass {rep}/{TIMING_REPEATS}: {n_new} artefacts timed")

## 8. Stage E — tables for the paper

In [ ]:
def fmt_pm(m, s, n):
    return f"{100 * m:.1f} ± {100 * s:.1f}" if n > 1 else f"{100 * m:.1f}"


def df_to_md(df):
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    lines += ["| " + " | ".join("" if (isinstance(v, float) and math.isnan(v)) else str(v) for v in r) + " |"
              for r in df.itertuples(index=False)]
    return "\n".join(lines) + "\n"


def save_table(df, name):
    df.to_csv(os.path.join(REPORT_DIR, f"{name}.csv"), index=False)
    with open(os.path.join(REPORT_DIR, f"{name}.md"), "w", encoding="utf-8") as f:
        f.write(df_to_md(df))


# ---- quality over seeds (GPU evaluation for transformers) ----
runs = []
for ds in DATA:
    for name in list(CLASSICAL_GRIDS) + list(TRANSFORMERS):
        for d in sorted(__import__("glob").glob(os.path.join(RUNS_DIR, ds, name, "seed*", "metrics.json"))):
            m = json.load(open(d))
            runs.append({"dataset": ds, "model": name, "seed": m["seed"], "macro_f1": m["macro_f1"], "accuracy": m["accuracy"],
                         **{f"f1_{k}": v for k, v in m["per_class_f1"].items()}})
RUNS = pd.DataFrame(runs)
quality = []
if len(RUNS):
    for (ds, name), g in RUNS.groupby(["dataset", "model"], sort=False):
        row = {"dataset": ds, "model": DISPLAY[name], "family": FAMILY[name], "seeds": len(g),
               "macro-F1 (%)": fmt_pm(g["macro_f1"].mean(), g["macro_f1"].std(ddof=1), len(g)),
               "accuracy (%)": fmt_pm(g["accuracy"].mean(), g["accuracy"].std(ddof=1), len(g))}
        for c in [c for c in g.columns if c.startswith("f1_")]:
            row[f"F1 {c[3:]} (%)"] = round(100 * g[c].mean(), 1)
        quality.append(row)
QUALITY = pd.DataFrame(quality)
save_table(QUALITY, "table_quality")
print("QUALITY (mean ± sd over seeds)\n", QUALITY.to_string(index=False) if len(QUALITY) else "no runs yet")

# ---- deployment: quality and cost of the same artefact on CPU ----
import glob as _glob, re as _re2
LEGACY_FORMATS = {"ONNX int8 per-channel": "ONNX int8 per-channel (no RR)"}   # earlier variant, kept for the table only
bench = []
for ds in DATA:
    for p in sorted(_glob.glob(os.path.join(BENCH_DIR, ds, "*__*.json"))):
        if p.endswith(".spec.json") or _re2.search(r"\.t\d+\.json$", p):
            continue
        r = json.load(open(p, encoding="utf-8"))
        r.pop("preds", None)
        passes = [r] + [json.load(open(t, encoding="utf-8")) for t in sorted(_glob.glob(p.replace(".json", ".t*.json")))]
        p50s = [x["p50_ms"] for x in passes]
        r["n_passes"] = len(passes)
        r["p50_spread_pct"] = (100 * (max(p50s) - min(p50s)) / float(np.median(p50s)) if len(passes) > 1
                               else r.get("p50_spread_pct", np.nan))
        for k in ("p50_ms", "p95_ms", "load_s", "peak_rss_mb"):   # median over passes
            r[k] = float(np.median([x[k] for x in passes]))
        bench.append(r)
BENCH = pd.DataFrame(bench)
if len(BENCH):
    native = BENCH[BENCH["format"].isin(["scikit-learn", "PyTorch fp32"])].set_index(["dataset", "model"])
    BENCH["delta_f1_vs_native"] = [r.macro_f1 - native["macro_f1"].get((r.dataset, r.model), np.nan) for r in BENCH.itertuples()]
    # ratios are what transfers across hardware; absolute ms depend on the Colab CPU
    fastest = BENCH.groupby("dataset")["p50_ms"].transform("min")
    BENCH["x_fastest"] = BENCH["p50_ms"] / fastest
    BENCH["speedup_vs_pytorch"] = [native["p50_ms"].get((r.dataset, r.model), np.nan) / r.p50_ms
                                   if r.model in TRANSFORMERS else np.nan for r in BENCH.itertuples()]
    DEPLOY = pd.DataFrame({
        "dataset": BENCH["dataset"], "model": BENCH["model"].map(DISPLAY), "format": BENCH["format"].replace(LEGACY_FORMATS),
        "macro-F1 (%)": (100 * BENCH["macro_f1"]).round(1), "Δ vs native (pts)": (100 * BENCH["delta_f1_vs_native"]).round(2),
        "size (MB)": BENCH["size_mb"].round(1), "peak RSS (MB)": BENCH["peak_rss_mb"].round(0),
        "cold start (s)": BENCH["load_s"].round(2), "p50 (ms)": BENCH["p50_ms"].round(2), "p95 (ms)": BENCH["p95_ms"].round(2),
        "timing passes": BENCH["n_passes"], "p50 spread across passes (%)": BENCH["p50_spread_pct"].round(1),
        "p50 × fastest": BENCH["x_fastest"].round(1), "speed-up vs PyTorch fp32": BENCH["speedup_vs_pytorch"].round(1),
        "throughput (texts/s)": BENCH["throughput_per_s"].round(1), "CPU ms/text": BENCH["cpu_ms_per_text"].round(2)})
    print("CPU flags during the benchmark:", sorted({tuple(x) for x in BENCH.get("cpu_flags", pd.Series(dtype=object)).dropna()}))
    DEPLOY = DEPLOY.sort_values(["dataset", "p50 (ms)"]).reset_index(drop=True)
    save_table(DEPLOY, "table_deployment")
    print(f"\nDEPLOYMENT ({BENCH_THREADS} thread, CPU: {BENCH['cpu'].iloc[0]})\n", DEPLOY.to_string(index=False))
else:
    DEPLOY = pd.DataFrame()
    print("\nno benchmark results yet (stage D)")

### Pareto frontier and deployment scenarios

A configuration is **Pareto-optimal** if no other configuration is at least as fast (or as small) *and* at least as accurate, with one strictly better. The scenario table picks, for each deployment budget, the most accurate configuration that fits it.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, LogLocator

DATASET_DISPLAY = {"twifil_clean": "TWIFIL (cleaned split)", "twifil_official": "TWIFIL (official split)",
                   "youtube": "Algerian Dialect YouTube", "narabizi": "NArabizi (Latin script)"}

# colours: validated categorical slots 1–3 (all-pairs CVD-safe); shape carries the format
FAMILY_COLOUR = {"Classical (TF-IDF)": "#2a78d6", "Multilingual small transformer": "#eb6834", "Arabic/dialect BERT": "#1baf7a"}
FORMAT_MARKER = {"scikit-learn": "o", "PyTorch fp32": "o", "ONNX fp32": "s", "ONNX int8": "^", "ONNX int8 per-channel+RR": "D"}
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3df"


def pareto_mask(cost, quality):
    cost, quality = np.asarray(cost), np.asarray(quality)
    keep = np.ones(len(cost), bool)
    for i in range(len(cost)):
        dominated = (cost <= cost[i]) & (quality >= quality[i]) & ((cost < cost[i]) | (quality > quality[i]))
        keep[i] = not dominated.any()
    return keep


def short_label(r):
    return DISPLAY[r.model] + {"ONNX fp32": " (ONNX)", "ONNX int8": " (int8)", "ONNX int8 per-channel+RR": " (int8-pc-RR)"}.get(r.format, "")


LABEL_OFFSETS = [(6, 5, "left"), (6, -12, "left"), (-6, 5, "right"), (-6, -12, "right"),
                 (6, 15, "left"), (-6, 15, "right"), (6, -22, "left"), (-6, -22, "right")]


def place_labels(fig, jobs, points):
    """Greedy placement: first offset whose text box hits no other label, no marker and stays inside the axes."""
    from matplotlib.transforms import Bbox
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    placed = {}
    for ax, txt, x, y in jobs:
        boxes = placed.setdefault(ax, [])
        marks = [Bbox.from_bounds(px - 6, py - 6, 12, 12) for px, py in ax.transData.transform(points[ax])]
        frame = ax.get_window_extent(renderer)
        chosen = None
        for dx, dy, ha in LABEL_OFFSETS:
            t = ax.annotate(txt, (x, y), xytext=(dx, dy), textcoords="offset points", fontsize=7, color=INK_2, ha=ha)
            bb = t.get_window_extent(renderer)
            inside = frame.x0 <= bb.x0 and bb.x1 <= frame.x1 and frame.y0 <= bb.y0 and bb.y1 <= frame.y1
            if inside and not any(bb.overlaps(o) for o in boxes + marks):
                chosen = bb
                break
            t.remove()
        if chosen is None:                               # nothing free: fall back to the first offset
            dx, dy, ha = LABEL_OFFSETS[0]
            t = ax.annotate(txt, (x, y), xytext=(dx, dy), textcoords="offset points", fontsize=7, color=INK_2, ha=ha)
            chosen = t.get_window_extent(renderer)
        boxes.append(chosen)


FIG = BENCH[~BENCH["format"].isin(list(LEGACY_FORMATS))] if len(BENCH) else BENCH   # legacy variant: table only
if len(FIG):
    dsets = [d for d in DATA if d in set(FIG["dataset"])]
    axes_spec = [("p50_ms", "Latency p50, batch 1 (ms, log scale)"), ("size_mb", "Size on disk (MB, log scale)")]
    fig_h = 3.3 * len(dsets) + 0.8                      # + room for the legend
    fig, axes = plt.subplots(len(dsets), 2, figsize=(10, fig_h), squeeze=False)
    frontier_rows, label_jobs, points = [], [], {}
    for i, ds in enumerate(dsets):
        b = FIG[FIG["dataset"] == ds].reset_index(drop=True)
        for j, (col, xlabel) in enumerate(axes_spec):
            ax = axes[i][j]
            mask = pareto_mask(b[col], b["macro_f1"])
            f = b[mask].sort_values(col)
            ax.step(f[col], 100 * f["macro_f1"], where="post", color=INK_2, lw=1.2, zorder=1)
            for r in b.itertuples():
                ax.scatter(getattr(r, col), 100 * r.macro_f1, s=64, marker=FORMAT_MARKER[r.format],
                           color=FAMILY_COLOUR[FAMILY[r.model]], edgecolor="white", linewidth=1.5, zorder=3)
            points[ax] = [(getattr(r, col), 100 * r.macro_f1) for r in b.itertuples()]
            label_jobs += [(ax, short_label(r), getattr(r, col), 100 * r.macro_f1) for r in f.itertuples()]  # frontier only
            ax.set_xscale("log")
            ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
            ax.xaxis.set_minor_locator(LogLocator(base=10, subs=(2, 5)))
            ax.xaxis.set_minor_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
            ax.set_xlabel(xlabel, fontsize=8, color=INK_2)
            ax.set_ylabel("Macro-F1 (%)", fontsize=8, color=INK_2)
            ax.set_title(DATASET_DISPLAY.get(ds, ds), fontsize=9, color=INK, loc="left")
            ax.grid(True, which="major", color=GRID, lw=0.8)
            ax.set_axisbelow(True)
            ax.tick_params(which="both", labelsize=7, colors=INK_2)
            for s in ("top", "right"):
                ax.spines[s].set_visible(False)
            for s in ("left", "bottom"):
                ax.spines[s].set_color(GRID)
            for r in f.itertuples():
                frontier_rows.append({"dataset": ds, "axis": "latency" if col == "p50_ms" else "size",
                                      "model": DISPLAY[r.model], "format": r.format, "macro-F1 (%)": round(100 * r.macro_f1, 1),
                                      "p50 (ms)": round(r.p50_ms, 2), "size (MB)": round(r.size_mb, 1)})
    fam_handles = [Line2D([], [], ls="", marker="o", ms=7, color=c, markeredgecolor="white", label=k) for k, c in FAMILY_COLOUR.items()]
    fmt_handles = [Line2D([], [], ls="", marker=m, ms=7, color=INK_2, markeredgecolor="white", label=k)
                   for k, m in (("native (scikit-learn / PyTorch)", "o"), ("ONNX fp32", "s"), ("ONNX int8", "^"),
                                ("ONNX int8 per-channel + reduce-range", "D"))]
    fr_handle = [Line2D([], [], color=INK_2, lw=1.2, label="Pareto frontier")]
    fig.tight_layout(rect=(0, 0.8 / fig_h, 1, 1))
    fig.legend(handles=fam_handles + fmt_handles + fr_handle, loc="lower center", ncol=4, fontsize=7.5, frameon=False,
               bbox_to_anchor=(0.5, 0.0))
    place_labels(fig, label_jobs, points)
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(REPORT_DIR, f"figure_pareto.{ext}"), dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    save_table(pd.DataFrame(frontier_rows), "table_pareto_frontier")

    # ---- deployment scenarios ----
    scen = []
    for ds in dsets:
        b = FIG[FIG["dataset"] == ds]
        for sname, (max_mb, max_p95) in SCENARIOS.items():
            ok = b[(b["size_mb"] <= max_mb) & (b["p95_ms"] <= max_p95)]
            if len(ok):
                r = ok.sort_values(["macro_f1", "p95_ms"], ascending=[False, True]).iloc[0]
                scen.append({"dataset": ds, "scenario": sname, "best configuration": f"{DISPLAY[r.model]} ({r.format})",
                             "macro-F1 (%)": round(100 * r.macro_f1, 1), "size (MB)": round(r.size_mb, 1),
                             "p95 (ms)": round(r.p95_ms, 2), "candidates that fit": len(ok)})
            else:
                scen.append({"dataset": ds, "scenario": sname, "best configuration": "none fits", "candidates that fit": 0})
    SCEN = pd.DataFrame(scen)
    save_table(SCEN, "table_scenarios")
    print(SCEN.to_string(index=False))

### TWIFIL: how much of the official score comes from the URL-only shortcut?

Notebook 01 found that ≈14 % of TWIFIL tweets contain no text once mentions and URLs are removed, and ≈98 % of those are labelled *neutral*. This cell splits the **official** test set into those tweets and the rest, and reports each model's accuracy on both parts, next to a one-line rule (*URL/mention only → neutral*).

In [ ]:
import re as _re
_URL_RE = _re.compile(r"(https?://\S+|www\.\S+)", _re.IGNORECASE)
_MEN_RE = _re.compile(r"@\w+:?")


def no_text(t):          # same definition as has_content() in notebook 01
    s = _MEN_RE.sub(" ", _URL_RE.sub(" ", _re.sub(r"^\s*RT\b", " ", str(t))))
    s = _re.sub(r"#", " ", s)
    return not _re.search("[A-Za-zÀ-ɏء-غف-يٱ-ۓۺ-ۿ]"
                          "|[\U0001F1E6-\U0001FAFF☀-➿]|[:;=][-']?[()\\[\\]DPp/\\\\|]|<3", s)


if "twifil_official" in DATA and len(RUNS):
    te = DATA["twifil_official"]["test"].copy()
    te["no_text"] = te["text"].map(no_text)
    rows = [{"model": "Rule: URL/mention only → neutral", "seeds": "-",
             "acc no-text part (%)": round(100 * (te.loc[te.no_text, "label"] == 1).mean(), 1),
             "acc text part (%)": np.nan, "acc overall (%)": np.nan}]
    for name in list(CLASSICAL_GRIDS) + list(TRANSFORMERS):
        accs = []
        for p in sorted(__import__("glob").glob(os.path.join(RUNS_DIR, "twifil_official", name, "seed*", "test_preds.csv"))):
            pr = pd.read_csv(p).merge(te[["id", "no_text"]], on="id")
            ok = pr["label"] == pr["pred"]
            accs.append((ok[pr.no_text].mean(), ok[~pr.no_text].mean(), ok.mean()))
        if accs:
            a = np.mean(accs, axis=0)
            rows.append({"model": DISPLAY[name], "seeds": len(accs), "acc no-text part (%)": round(100 * a[0], 1),
                         "acc text part (%)": round(100 * a[1], 1), "acc overall (%)": round(100 * a[2], 1)})
    SHORTCUT = pd.DataFrame(rows)
    save_table(SHORTCUT, "table_twifil_shortcut")
    print(f"Official TWIFIL test: {int(te.no_text.sum())} of {len(te)} tweets ({100 * te.no_text.mean():.1f} %) have no text")
    print(SHORTCUT.to_string(index=False))
    print("\nPublished reference (same split, accuracy): DziriBERT 80.5, MARBERT 80.5, CAMeLBERT-mix 77.7")

### Bootstrap 95 % confidence intervals (test macro-F1, benchmarked artefacts)

In [ ]:
def bootstrap_ci(y, p, n=N_BOOTSTRAP, seed=0):
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    labs = sorted(set(y))
    vals = [f1_score(y[idx], p[idx], labels=labs, average="macro", zero_division=0)
            for idx in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(vals, [2.5, 97.5])


def paired_diff_ci(y, pa, pb, n=N_BOOTSTRAP, seed=0):
    rng = np.random.default_rng(seed)
    y, pa, pb = map(np.asarray, (y, pa, pb))
    labs = sorted(set(y))
    f = lambda idx, p: f1_score(y[idx], p[idx], labels=labs, average="macro", zero_division=0)
    d = [f(idx, pa) - f(idx, pb) for idx in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(d, [2.5, 97.5])


ci_rows = []
for ds in DATA:
    y = DATA[ds]["test"]["label"].values
    preds = {}
    for p in sorted(__import__("glob").glob(os.path.join(BENCH_DIR, ds, "*__*.json"))):
        if p.endswith(".spec.json") or _re2.search(r"\.t\d+\.json$", p):     # skip specs and timing-only passes
            continue
        r = json.load(open(p, encoding="utf-8"))
        if r["format"] in ("scikit-learn", "PyTorch fp32"):
            preds[r["model"]] = np.asarray(r["preds"])
    if not preds:
        continue
    f1s = {m: f1_score(y, pr, average="macro") for m, pr in preds.items()}
    for m, pr in preds.items():
        lo, hi = bootstrap_ci(y, pr)
        ci_rows.append({"dataset": ds, "model": DISPLAY[m], "macro-F1 (%)": round(100 * f1s[m], 1),
                        "95% CI": f"[{100 * lo:.1f}, {100 * hi:.1f}]"})
    cls = [m for m in preds if m in CLASSICAL_GRIDS]
    trf = [m for m in preds if m in TRANSFORMERS]
    if cls and trf:
        bt, bc = max(trf, key=f1s.get), max(cls, key=f1s.get)
        lo, hi = paired_diff_ci(y, preds[bt], preds[bc])
        ci_rows.append({"dataset": ds, "model": f"Δ {DISPLAY[bt]} − {DISPLAY[bc]}",
                        "macro-F1 (%)": round(100 * (f1s[bt] - f1s[bc]), 1), "95% CI": f"[{100 * lo:.1f}, {100 * hi:.1f}]"})
CI = pd.DataFrame(ci_rows)
save_table(CI, "table_bootstrap_ci")
print(CI.to_string(index=False) if len(CI) else "no benchmark results yet")

## 9. Provenance

In [ ]:
prov = {"created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "smoke_test": SMOKE_TEST, "run_stages": RUN_STAGES, "bench_cpu": cpu_name(), "cpu_flags": cpu_flags(), "gpu": GPU_NAME,
        "versions": {"python": platform.python_version(), "torch": getattr(torch, "__version__", None),
                     "transformers": __import__("transformers").__version__ if torch is not None else None,
                     "onnxruntime": __import__("onnxruntime").__version__, "scikit-learn": sklearn.__version__},
        "config": {"TRANSFORMERS": TRANSFORMERS, "SEEDS": SEEDS, "EPOCHS": EPOCHS, "PATIENCE": PATIENCE,
                   "BATCH_SIZE": BATCH_SIZE, "MAX_LEN_CAP": MAX_LEN_CAP, "CLASSICAL_GRIDS": CLASSICAL_GRIDS,
                   "BENCH_THREADS": BENCH_THREADS, "N_SINGLE": N_SINGLE, "BENCH_ROUNDS": BENCH_ROUNDS, "BENCH_BATCH": BENCH_BATCH,
                   "TIMING_REPEATS": TIMING_REPEATS, "N_TIMING": N_TIMING,
                   "SCENARIOS": {k: list(v) for k, v in SCENARIOS.items()}},
        "runs_done": int(len(RUNS)), "artefacts_benchmarked": int(len(BENCH))}
with open(os.path.join(REPORT_DIR, "provenance_02.json"), "w") as f:
    json.dump(prov, f, indent=2, default=str)
todo_b = sum(1 for s in SEEDS for ds in DATA for m in TRANSFORMERS if not os.path.exists(os.path.join(run_dir(ds, m, s), "metrics.json")))
print(f"runs done: {len(RUNS)} | transformer runs still to do: {todo_b} | artefacts benchmarked: {len(BENCH)}")
print("reports in", REPORT_DIR)

## Notes for the paper

- **Two quality numbers per transformer, on purpose.** `table_quality` is mean ± sd over seeds (GPU evaluation); `table_deployment` is the benchmarked seed-0 artefact on CPU, per format. The Δ column isolates what ONNX export and int8 quantisation cost in accuracy.
- **No physical edge device.** All cost figures come from one thread of the Colab virtual CPU. Run stage D in one sitting and one runtime type; the CPU model is saved in every result. Claim the ranking and the ratios (`p50 × fastest`, `speed-up vs PyTorch fp32`), not the absolute milliseconds; `p50 spread over rounds` shows how stable the timing was. Size on disk and peak memory do not depend on CPU speed and can be stated as measured.
- **Budget tiers** in the scenario table are assumptions stated on the benchmark CPU; give the thresholds explicitly in the paper. Lead with the Pareto frontier (threshold-free) and treat the scenario table as an illustration: a model a few MB or ms over a threshold flips the choice.
- **INT8 on x86.** ONNX Runtime's dynamic quantisation uses U8S8 kernels (VPMADDUBSW) on AVX2/AVX512 CPUs; without VNNI the 16-bit intermediate can saturate. The ORT documentation recommends `reduce_range` (and says per-channel needs it on these CPUs). The CPU flags are recorded in every result; report them with the int8 numbers.
- Suggested wording (experimental setup / threats to validity): *"Latency, throughput and memory were measured on a single thread of a cloud virtual CPU ({CPU model}), each model in a fresh process, with the single-text timing repeated in three rounds. No physical edge device was used; absolute latencies on embedded hardware will differ, so we base our conclusions on relative costs between models measured under identical conditions, and on size and memory footprint, which are hardware-independent."*
- `table_twifil_shortcut` quantifies how much of the official TWIFIL accuracy comes from URL-only tweets; pair it with the audit table from notebook 01.